In [3]:
import os
import shutil
import pandas as pd
from pandas.errors import EmptyDataError, ParserError

base_p = r"E:\Frame_Num\LaneCount\location"

for fn in os.listdir(base_p):
    fd_p = os.path.join(base_p, fn)

    if not os.path.isdir(fd_p):
        continue

    csvs = [x for x in os.listdir(fd_p) if x.lower().endswith(".csv")]

    if len(csvs) == 0:
        continue

    csv_p = os.path.join(fd_p, csvs[0])

    try:
        df = pd.read_csv(csv_p)
    except (EmptyDataError, ParserError, UnicodeDecodeError, OSError, ValueError) as e:
        # can't parse -> delete folder
        try:
            shutil.rmtree(fd_p)
            print("Deleted folder (CSV parse failed):", fd_p, "| error:", type(e).__name__)
        except Exception as e2:
            print("FAILED to delete folder:", fd_p, "| error:", type(e2).__name__)
        continue

    # Ensure lane_count exists
    if "lane_count" not in df.columns:
        try:
            shutil.rmtree(fd_p)
            print("Deleted folder (missing lane_count col):", fd_p)
        except Exception as e2:
            print("FAILED to delete folder:", fd_p, "| error:", type(e2).__name__)
        continue

    # drop N/A lane_count rows (handle NaN + string variants)
    s = df["lane_count"].astype(str).str.strip().str.lower()
    bad = df["lane_count"].isna() | s.isin(["n/a", "na", "none", "nan", ""])

    df2 = df[~bad]

    if len(df2) == 0:
        try:
            shutil.rmtree(fd_p)
            print("Deleted folder (all rows N/A):", fd_p)
        except Exception as e2:
            print("FAILED to delete folder:", fd_p, "| error:", type(e2).__name__)
    else:
        try:
            df2.to_csv(csv_p, index=False)
            print("Cleaned CSV:", csv_p, "| kept rows:", len(df2), "/", len(df))
        except Exception as e:
            # if we can't write back, treat as bad folder
            try:
                shutil.rmtree(fd_p)
                print("Deleted folder (write failed):", fd_p, "| error:", type(e).__name__)
            except Exception as e2:
                print("FAILED to delete folder:", fd_p, "| error:", type(e2).__name__)


Cleaned CSV: E:\Frame_Num\LaneCount\location\20250826_31302PMByGPSMapCamera\20250826_31302PMByGPSMapCamera_Result.csv | kept rows: 161 / 161
Cleaned CSV: E:\Frame_Num\LaneCount\location\20250826_31410PMByGPSMapCamera\20250826_31410PMByGPSMapCamera_Result.csv | kept rows: 271 / 271
Cleaned CSV: E:\Frame_Num\LaneCount\location\20250826_31426PMByGPSMapCamera\20250826_31426PMByGPSMapCamera_Result.csv | kept rows: 880 / 880
Cleaned CSV: E:\Frame_Num\LaneCount\location\20250826_31635PMByGPSMapCamera\20250826_31635PMByGPSMapCamera_Result.csv | kept rows: 1517 / 1517
Cleaned CSV: E:\Frame_Num\LaneCount\location\20250826_32642PMByGPSMapCamera\20250826_32642PMByGPSMapCamera_Result.csv | kept rows: 10860 / 10860
Cleaned CSV: E:\Frame_Num\LaneCount\location\20250826_34402PMByGPSMapCamera\20250826_34402PMByGPSMapCamera_Result.csv | kept rows: 2260 / 2260
Cleaned CSV: E:\Frame_Num\LaneCount\location\20250826_34712PMByGPSMapCamera\20250826_34712PMByGPSMapCamera_Result.csv | kept rows: 1909 / 1909
Cle

In [4]:
import os
import pandas as pd
from pandas.errors import EmptyDataError, ParserError

base_p = r"E:\Frame_Num\LaneCount\location"

bad_csvs = []      # CSVs that contain any N/A in lane_count
fail_csvs = []     # CSVs we couldn't read/parse
miss_csvs = []     # CSVs missing lane_count column

for fn in os.listdir(base_p):
    fd_p = os.path.join(base_p, fn)
    if not os.path.isdir(fd_p):
        continue

    for cf in os.listdir(fd_p):
        if not cf.lower().endswith(".csv"):
            continue

        csv_p = os.path.join(fd_p, cf)

        try:
            df = pd.read_csv(csv_p)
        except (EmptyDataError, ParserError, UnicodeDecodeError, OSError, ValueError) as e:
            fail_csvs.append((csv_p, type(e).__name__))
            continue

        if "lane_count" not in df.columns:
            miss_csvs.append(csv_p)
            continue

        s = df["lane_count"].astype(str).str.strip().str.lower()
        bad = df["lane_count"].isna() | s.isin(["n/a", "na", "none", "nan", ""])
        c_bad = int(bad.sum())

        if c_bad > 0:
            bad_csvs.append((csv_p, c_bad, len(df)))

print("\n=== CSVs with any N/A lane_count ===")
if len(bad_csvs) == 0:
    print("None found.")
else:
    for p, nb, nt in bad_csvs:
        print(p, "| N/A rows:", nb, "/", nt)

print("\n=== CSVs that failed to parse ===")
if len(fail_csvs) == 0:
    print("None.")
else:
    for p, err in fail_csvs:
        print(p, "| error:", err)

print("\n=== CSVs missing lane_count column ===")
if len(miss_csvs) == 0:
    print("None.")
else:
    for p in miss_csvs:
        print(p)



=== CSVs with any N/A lane_count ===
None found.

=== CSVs that failed to parse ===
None.

=== CSVs missing lane_count column ===
None.


In [8]:
import os
import shutil
import re

loc_p = r"E:\Frame_Num\LaneCount\location"
frm_p = r"E:\Frame_Num\RedextractedFrames 3\RedextractedFrames"

# extract YYYYMMDD_NNNNN
def key_id(name):
    m = re.match(r"(\d{8}_\d{5})", name)
    return m.group(1) if m else None

# collect valid keys from LaneCount/location
keep = set()
for fn in os.listdir(loc_p):
    fp = os.path.join(loc_p, fn)
    if not os.path.isdir(fp):
        continue

    k = key_id(fn)
    if k:
        keep.add(k)

print("Keys kept:", len(keep))

# delete RedextractedFrames folders without match
del_ct = 0
for fn in os.listdir(frm_p):
    fp = os.path.join(frm_p, fn)
    if not os.path.isdir(fp):
        continue

    k = key_id(fn)

    if (k is None) or (k not in keep):
        try:
            shutil.rmtree(fp)
            del_ct = del_ct + 1
            print("Deleted (no match):", fp)
        except Exception as e:
            print("FAILED to delete:", fp, "| error:", type(e).__name__)

print("\nDone. Deleted folders:", del_ct)


Keys kept: 13
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250826_30837_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250826_30958_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250826_32159_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250826_32351_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250826_33913_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250828_32715_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250828_32823_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250828_33133_RedExtracted
Deleted (no match): E:\Frame_Num\RedextractedFrames 3\RedextractedFrames\20250828_83117_RedExtracted

Done. Deleted folders: 9


In [11]:
import os
import pandas as pd
from collections import Counter
from pandas.errors import EmptyDataError, ParserError

base_p = r"E:\Frame_Num\LaneCount\location"

total = Counter()

for fn in os.listdir(base_p):
    fd_p = os.path.join(base_p, fn)
    if not os.path.isdir(fd_p):
        continue

    for cf in os.listdir(fd_p):
        if not cf.lower().endswith(".csv"):
            continue

        csv_p = os.path.join(fd_p, cf)

        try:
            df = pd.read_csv(csv_p)
        except (EmptyDataError, ParserError, UnicodeDecodeError, OSError, ValueError):
            continue

        if "lane_count" not in df.columns:
            continue

        lc = pd.to_numeric(df["lane_count"], errors="coerce")
        lc = lc[lc.notna()]   # drop NaNs

        total.update(lc.tolist())

print("\n=== TOTAL lane_count distribution (including 2) ===")
for k in sorted(total):
    print(f"lane_count={int(k)} : {total[k]}")



=== TOTAL lane_count distribution (including 2) ===
lane_count=1 : 339
lane_count=2 : 30222
lane_count=3 : 575
lane_count=4 : 271


In [13]:
import os
import re
import shutil
import pandas as pd
from pandas.errors import EmptyDataError, ParserError
from tqdm import tqdm

img_root = r"E:\Frame_Num\RedextractedFrames 3\RedextractedFrames"
csv_root = r"E:\Frame_Num\LaneCount\location"
out_root = r"E:\Frame_Num\MatchedFrames"
os.makedirs(out_root, exist_ok=True)

def key_id(name):
    m = re.match(r"(\d{8}_\d{5})", name)   # 20250826_31302
    return m.group(1) if m else None

def img_frame_num(jpg_name):
    # red_frame_000006.jpg -> 6
    m = re.search(r"(\d+)(?=\.jpg$)", jpg_name.lower())
    return int(m.group(1)) if m else None

# -------------------------------------------------
# Build index of available JPGs
# -------------------------------------------------
img_map = {}

for sf in tqdm(os.listdir(img_root), desc="Indexing image folders"):
    sp = os.path.join(img_root, sf)
    if not os.path.isdir(sp):
        continue

    k = key_id(sf)
    if not k:
        continue

    d = {}
    for f in os.listdir(sp):
        if not f.lower().endswith(".jpg"):
            continue
        fr = img_frame_num(f)
        if fr is not None:
            d[fr] = os.path.join(sp, f)

    img_map[k] = d

# -------------------------------------------------
# Match CSV frames → copy & rename images
# -------------------------------------------------
copy_ct = 0
miss_ct = 0

for sf in tqdm(os.listdir(csv_root), desc="Processing CSV folders"):
    sp = os.path.join(csv_root, sf)
    if not os.path.isdir(sp):
        continue

    k = key_id(sf)
    if not k:
        continue

    csvs = [x for x in os.listdir(sp) if x.lower().endswith(".csv")]
    if len(csvs) == 0:
        continue

    csv_p = os.path.join(sp, csvs[0])

    try:
        df = pd.read_csv(csv_p)
    except (EmptyDataError, ParserError, UnicodeDecodeError, OSError, ValueError):
        continue

    need = {"video_name", "frame_number", "lane_count"}
    if not need.issubset(df.columns):
        continue

    if k not in img_map:
        continue

    df["frame_number"] = pd.to_numeric(df["frame_number"], errors="coerce")
    df["lane_count"] = pd.to_numeric(df["lane_count"], errors="coerce")
    df = df[df["frame_number"].notna() & df["lane_count"].notna()]

    for _, r in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Copying frames [{k}]",
        leave=False
    ):
        vn = str(r["video_name"]).strip()
        fr = int(r["frame_number"])
        lc = int(r["lane_count"])

        if fr in img_map[k]:
            src = img_map[k][fr]
            dst_name = f"{vn}_{fr}_{lc}.jpg"
            dst = os.path.join(out_root, dst_name)

            # avoid overwriting
            if os.path.exists(dst):
                base, ext = os.path.splitext(dst_name)
                aa = 2
                while os.path.exists(os.path.join(out_root, f"{base}_{aa}{ext}")):
                    aa = aa + 1
                dst = os.path.join(out_root, f"{base}_{aa}{ext}")

            shutil.copy2(src, dst)
            copy_ct = copy_ct + 1
        else:
            miss_ct = miss_ct + 1

print("\nDone.")
print("Copied:", copy_ct)
print("Missing (CSV frame not found as JPG):", miss_ct)
print("Output folder:", out_root)


Copying frames [20250826_32642]:  22%|█████▌                    | 2344/10860 [00:03<00:14, 582.76it/s]


Copying frames [20250826_34712]:   7%|██                          | 141/1909 [00:00<00:02, 712.94it/s]


Copying frames [20250828_83400]:  64%|█████████████████▎         | 2608/4058 [00:03<00:02, 693.62it/s]


Copying frames [20250828_84212]:  64%|█████████████████▎         | 4653/7235 [00:06<00:03, 687.23it/s]


Copying frames [20250828_84212]:  74%|████████████████████▊       | 5363/7235 [00:19<00:34, 54.51it/s]


Copying frames [20250828_84212]:  81%|██████████████████████▋     | 5876/7235 [00:28<00:23, 58.98it/s]


Copying frames [20250828_84212]:  89%|████████████████████████▊   | 6405/7235 [00:38<00:20, 41.33it/s]


Copying frames [20250828_84212]:  95%|██████████████████████████▋ | 6905/7235 [00:47<00:05, 63.34it/s]


Processing CSV folders: 100%|█████████████████████████████████████████| 13/13 [01:19<00:00,  6.10s/it]


Done.
Copied: 24410
Missing (CSV frame not found as JPG): 6997
Output folder: E:\Frame_Num\MatchedFrames


In [14]:
import os
import re
import shutil
import pandas as pd
from pandas.errors import EmptyDataError, ParserError
from tqdm import tqdm

img_root = r"E:\Frame_Num\RedextractedFrames 3\RedextractedFrames"
csv_root = r"E:\Frame_Num\LaneCount\location"
out_root = r"E:\Frame_Num\MatchedFrames"
unm_root = r"E:\Frame_Num\Unmatch"

os.makedirs(out_root, exist_ok=True)
os.makedirs(unm_root, exist_ok=True)

def key_id(name):
    m = re.match(r"(\d{8}_\d{5})", name)
    return m.group(1) if m else None

def img_frame_num(jpg_name):
    m = re.search(r"(\d+)(?=\.jpg$)", jpg_name.lower())
    return int(m.group(1)) if m else None

# -------------------------------------------------
# Index all images
# -------------------------------------------------
img_map = {}
all_imgs = {}   # key -> frame -> path

for sf in tqdm(os.listdir(img_root), desc="Indexing image folders"):
    sp = os.path.join(img_root, sf)
    if not os.path.isdir(sp):
        continue

    k = key_id(sf)
    if not k:
        continue

    d = {}
    for f in os.listdir(sp):
        if not f.lower().endswith(".jpg"):
            continue
        fr = img_frame_num(f)
        if fr is not None:
            p = os.path.join(sp, f)
            d[fr] = p
            all_imgs[p] = False   # mark unused initially

    img_map[k] = d

# -------------------------------------------------
# Match CSV → copy matched images
# -------------------------------------------------
copy_ct = 0

for sf in tqdm(os.listdir(csv_root), desc="Processing CSV folders"):
    sp = os.path.join(csv_root, sf)
    if not os.path.isdir(sp):
        continue

    k = key_id(sf)
    if not k or k not in img_map:
        continue

    csvs = [x for x in os.listdir(sp) if x.lower().endswith(".csv")]
    if not csvs:
        continue

    csv_p = os.path.join(sp, csvs[0])

    try:
        df = pd.read_csv(csv_p)
    except (EmptyDataError, ParserError, UnicodeDecodeError, OSError, ValueError):
        continue

    if not {"video_name", "frame_number", "lane_count"}.issubset(df.columns):
        continue

    df["frame_number"] = pd.to_numeric(df["frame_number"], errors="coerce")
    df["lane_count"] = pd.to_numeric(df["lane_count"], errors="coerce")
    df = df[df["frame_number"].notna() & df["lane_count"].notna()]

    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Copying [{k}]", leave=False):
        fr = int(r["frame_number"])
        if fr not in img_map[k]:
            continue

        src = img_map[k][fr]
        all_imgs[src] = True   # mark as matched

        vn = str(r["video_name"]).strip()
        lc = int(r["lane_count"])

        dst_name = f"{vn}_{fr}_{lc}.jpg"
        dst = os.path.join(out_root, dst_name)

        if os.path.exists(dst):
            base, ext = os.path.splitext(dst_name)
            i = 2
            while os.path.exists(os.path.join(out_root, f"{base}_{i}{ext}")):
                i = i + 1
            dst = os.path.join(out_root, f"{base}_{i}{ext}")

        shutil.copy2(src, dst)
        copy_ct = copy_ct + 1

# -------------------------------------------------
# Move unmatched images
# -------------------------------------------------
unmatch_ct = 0
for p, used in tqdm(all_imgs.items(), desc="Moving unmatched images"):
    if not used:
        dst = os.path.join(unm_root, os.path.basename(p))
        if not os.path.exists(dst):
            shutil.copy2(p, dst)
        unmatch_ct = unmatch_ct + 1

print("\nDone.")
print("Matched images copied:", copy_ct)
print("Unmatched images copied:", unmatch_ct)
print("Matched folder:", out_root)
print("Unmatched folder:", unm_root)


Copying [20250826_32642]:  13%|████▍                            | 1447/10860 [00:02<00:17, 527.59it/s]


Copying [20250826_34402]:  36%|████████████▊                      | 824/2260 [00:01<00:01, 803.14it/s]


Copying [20250828_83400]:   4%|█▍                                 | 168/4058 [00:00<00:06, 566.75it/s]


Processing CSV folders:  85%|██████████████████████████████████▋      | 11/13 [00:29<00:05,  2.72s/it]


KeyboardInterrupt: 

In [ ]:
import os
import re
import shutil
from tqdm import tqdm

A_root = r"E:\Frame_Num\RedextractedFrames 3\RedextractedFrames"
B_root = r"E:\RedextractedFrames 3\RedextractedFrames"
unm_root = r"E:\Frame_Num\Unmatch"
os.makedirs(unm_root, exist_ok=True)

def key_id(name):
    m = re.match(r"(\d{8}_\d{5})", name)
    return m.group(1) if m else None

def copy_all_jpgs(folder_path, tag):
    global copy_ct
    imgs = [x for x in os.listdir(folder_path) if x.lower().endswith(".jpg")]
    for im in imgs:
        src = os.path.join(folder_path, im)

        # tag to avoid collisions + keep provenance
        dst_name = f"{tag}_{os.path.basename(folder_path)}_{im}"
        dst = os.path.join(unm_root, dst_name)

        # avoid overwrite if still collides
        if os.path.exists(dst):
            base, ext = os.path.splitext(dst_name)
            i = 2
            while os.path.exists(os.path.join(unm_root, f"{base}_{i}{ext}")):
                i = i + 1
            dst = os.path.join(unm_root, f"{base}_{i}{ext}")

        shutil.copy2(src, dst)
        copy_ct = copy_ct + 1

# --------- build maps: key -> folderpath ----------
A_map = {}
for fn in os.listdir(A_root):
    fp = os.path.join(A_root, fn)
    if os.path.isdir(fp):
        k = key_id(fn)
        if k:
            A_map[k] = fp

B_map = {}
for fn in os.listdir(B_root):
    fp = os.path.join(B_root, fn)
    if os.path.isdir(fp):
        k = key_id(fn)
        if k:
            B_map[k] = fp

A_keys = set(A_map.keys())
B_keys = set(B_map.keys())

only_A = sorted(list(A_keys - B_keys))
only_B = sorted(list(B_keys - A_keys))

copy_ct = 0
folder_ct = 0

# folders in A but not in B
for k in tqdm(only_A, desc="Copying A-only folders"):
    folder_ct = folder_ct + 1
    copy_all_jpgs(A_map[k], tag="Aonly")

# folders in B but not in A
for k in tqdm(only_B, desc="Copying B-only folders"):
    folder_ct = folder_ct + 1
    copy_all_jpgs(B_map[k], tag="Bonly")

print("\nDone.")
print("A folders:", len(A_keys), "| B folders:", len(B_keys))
print("A-only folders:", len(only_A), "| B-only folders:", len(only_B))
print("Total unmatched folders copied from:", folder_ct)
print("Total images copied to Unmatch:", copy_ct)
print("Unmatch folder:", unm_root)


Copying A-only folders: 0it [00:00, ?it/s]
Copying B-only folders:  89%|██████████████████████████████████████▏    | 8/9 [05:45<00:35, 35.65s/it]